# Package Serving Artifact

Purpose: take an existing staged run from `ml_model/model_registry/staging/`, load it using the same checkpoint pattern used in `edited.ipynb` and `evaluate.ipynb`, and export a self-contained local serving artifact into that run directory.

This notebook does not retrain. It only packages an already trained run by adding Hugging Face model/tokenizer assets and a small serving manifest.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ml_model").exists() and (candidate / "web_app").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ML_ROOT = REPO_ROOT / "ml_model"
MODEL_REGISTRY = ML_ROOT / "model_registry"
STAGING_DIR = MODEL_REGISTRY / "staging"
EVAL_DIR = MODEL_REGISTRY / "eval"
DEVICE = torch.device("cpu")

LABEL_NAMES = ["Code Injection", "Normal", "Other Attacks", "SQL Injection"]
MODEL_IDS = {
    "minilm": "nreimers/MiniLM-L6-H384-uncased",
    "distilbert": "distilbert-base-uncased",
    "bert-base": "bert-base-uncased",
}

print(f"Repo root      : {REPO_ROOT}")
print(f"Model registry : {MODEL_REGISTRY}")
print(f"Staging dir    : {STAGING_DIR}")
print(f"Eval dir       : {EVAL_DIR}")

In [ ]:
# Change these only if you want to package a different run.
MODEL_KEY = "distilbert"
RUN_DIR_NAME: str | None = None


def discover_latest_run(staging_dir: Path, model_key: str) -> Path:
    candidates = [
        path for path in staging_dir.iterdir()
        if path.is_dir() and path.name.startswith(model_key + "_")
    ]
    if not candidates:
        raise FileNotFoundError(f"No staged run found for {model_key} in {staging_dir}")
    candidates.sort(key=lambda path: path.name, reverse=True)
    return candidates[0]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def load_temperature(eval_dir: Path, model_key: str) -> float:
    if not eval_dir.exists():
        return 1.0
    for run_dir in sorted([path for path in eval_dir.iterdir() if path.is_dir()], key=lambda path: path.name, reverse=True):
        result_path = run_dir / f"eval_results_{model_key}_calibrated.json"
        if result_path.exists():
            return float(load_json(result_path).get("temperature", 1.0))
    return 1.0


run_dir = STAGING_DIR / RUN_DIR_NAME if RUN_DIR_NAME else discover_latest_run(STAGING_DIR, MODEL_KEY)
ckpt_path = run_dir / f"best_{MODEL_KEY}_ckpt.pt"
config_used = load_json(run_dir / "config_used.json")
model_id = config_used.get("model_id", MODEL_IDS[MODEL_KEY])
max_seq_len = int(config_used.get("max_seq_len", 128))
temperature = load_temperature(EVAL_DIR, MODEL_KEY)

assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

print(f"Run directory : {run_dir}")
print(f"Checkpoint    : {ckpt_path.name}")
print(f"Model ID      : {model_id}")
print(f"Max seq len   : {max_seq_len}")
print(f"Temperature   : {temperature:.6f}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(LABEL_NAMES),
)
state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
assert isinstance(state, dict), f"Expected state_dict, got {type(state)}"
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Checkpoint and tokenizer loaded successfully.")

In [ ]:
model.save_pretrained(run_dir)
tokenizer.save_pretrained(run_dir)

serving_manifest = {
    "exported_at": datetime.now().isoformat(timespec="seconds"),
    "source_notebooks": [
        "ml_model/edited.ipynb",
        "ml_model/evaluate.ipynb",
    ],
    "model_key": MODEL_KEY,
    "model_id": model_id,
    "run_dir": run_dir.name,
    "checkpoint_file": ckpt_path.name,
    "label_names": LABEL_NAMES,
    "max_seq_len": max_seq_len,
    "temperature": round(float(temperature), 6),
    "local_reload_expected": True,
}

(run_dir / "serving_manifest.json").write_text(
    json.dumps(serving_manifest, indent=2) + "\n",
    encoding="utf-8",
)

expected_files = [
    "config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "serving_manifest.json",
]
present = sorted(path.name for path in run_dir.iterdir())
missing = [name for name in expected_files if name not in present]

print("Run directory contents:")
for name in present:
    print(f"  - {name}")

if missing:
    print("\nExpected files still missing:")
    for name in missing:
        print(f"  - {name}")
else:
    print("\nExpected serving files are present.")

In [ ]:
reload_model = AutoModelForSequenceClassification.from_pretrained(
    run_dir,
    local_files_only=True,
)
reload_tokenizer = AutoTokenizer.from_pretrained(
    run_dir,
    local_files_only=True,
)
reload_model.to(DEVICE).eval()

sample = "SELECT * FROM users WHERE 1=1 --"
encoded = reload_tokenizer(
    sample,
    truncation=True,
    max_length=max_seq_len,
    return_tensors="pt",
    padding=True,
)

with torch.no_grad():
    logits = reload_model(**encoded).logits.float()

probs = torch.softmax(logits / float(temperature), dim=-1).squeeze().tolist()
pred_idx = int(torch.argmax(torch.tensor(probs)).item())

print(f"Sample prediction : {LABEL_NAMES[pred_idx]}")
print(f"Confidence        : {max(probs):.6f}")
print("Local-only reload succeeded.")

After this notebook succeeds, the selected run directory contains both the original checkpoint-based files and the extra Hugging Face serving files needed for local-only reload.

Next step after packaging: update the application loader to prefer `run_dir` local files instead of bootstrapping from `model_id`.